In [1]:
import h5py
import jax
jax.config.update("jax_enable_x64", True)
import numpy as np

## Load generated data

The data generated for the tests consists of true, time-delayed detector noise from O3a and an injected signal. Truth parameters are saved to facilitate tests.

In [2]:
arrays = {}

def collect(name, obj):
    if isinstance(obj, h5py.Dataset):
        arrays[name] = obj[...]   # loads into a NumPy array

with h5py.File("test_bbh_v3.h5", "r") as f:
    f.visititems(collect)

# Now `arrays` holds everything
for name, arr in arrays.items():
    print(f"{name}: shape={arr.shape}, dtype={arr.dtype}")

H1/psd: shape=(16385,), dtype=float64
H1/strain: shape=(32768,), dtype=float32
L1/psd: shape=(16385,), dtype=float64
L1/strain: shape=(32768,), dtype=float32
antenna/H1/fcross: shape=(), dtype=float64
antenna/H1/fplus: shape=(), dtype=float64
antenna/L1/fcross: shape=(), dtype=float64
antenna/L1/fplus: shape=(), dtype=float64
truth/chi1: shape=(), dtype=float32
truth/chi2: shape=(), dtype=float32
truth/chirp_mass: shape=(), dtype=float32
truth/dec: shape=(), dtype=float32
truth/distance: shape=(), dtype=float32
truth/inclination: shape=(), dtype=float32
truth/mass_1: shape=(), dtype=float32
truth/mass_2: shape=(), dtype=float32
truth/mass_ratio: shape=(), dtype=float32
truth/phi: shape=(), dtype=float32
truth/phic: shape=(), dtype=float32
truth/psi: shape=(), dtype=float32
truth/s1z: shape=(), dtype=float32
truth/s2z: shape=(), dtype=float32
truth/snr: shape=(), dtype=float32
truth/snr_H1: shape=(), dtype=float32
truth/snr_L1: shape=(), dtype=float32
truth/tc: shape=(), dtype=float64


## Plotting the inputs in time and frequency space

In [3]:
# set the interferometer here!!!
ifo = 'H1'

if ifo == 'H1':
    c = '#ee0000'
if ifo == 'L1':
    c= '#4ba6ff'

In [ ]:
from gwpy.timeseries import TimeSeries
strain_ts = TimeSeries(arrays[f"{ifo}/strain"], t0=0, sample_rate=4096)

plot = strain_ts.plot(
    title=f"{ifo} strain with realistic detector noise and injected synthetic signal",
    ylabel="Strain",
    color=c,
)
plot.show()


In [ ]:
from gwpy.frequencyseries import FrequencySeries
psd_fs = FrequencySeries(
    arrays[f"{ifo}/psd"],
    f0=0,
    df=1 / 8,
)
asd = psd_fs ** 0.5
white = strain_ts.whiten(asd=asd).bandpass(30, 400)

plot = white.plot(color=c)
ax = plot.gca()
ax.set_xlim(6.0, 8)


In [ ]:
qtrans = white.q_transform(
    frange=(20, 500),
    outseg=(6.25, 8),
    whiten=False,
)

plot = qtrans.plot(figsize=(8, 5), vmin=0, vmax=25)
ax = plot.gca()
ax.set_yscale("log")
ax.set_ylabel("Frequency [Hz]")
plot.colorbar(label="Normalized energy")
plot.show()


## Let's move to the fitting part

In [ ]:
import jax.numpy as jnp

psd_arr = jnp.asarray(np.asarray(psd_fs))
strain = jnp.asarray(np.asarray(strain_ts))


In [ ]:
# Convert truth to ripple's parameter order
Mc          = float(arrays["truth/chirp_mass"])
q           = float(arrays["truth/mass_ratio"])
eta         = q / (1 + q) ** 2
chi1        = float(arrays["truth/chi1"])
chi2        = float(arrays["truth/chi2"])
tc          = float(arrays["truth/tc"])
phic        = float(arrays["truth/phic"])
dist_Mpc    = float(arrays["truth/distance"])
inclination = float(arrays["truth/inclination"])

theta_truth = jnp.array(
    [Mc, eta, chi1, chi2, dist_Mpc, tc, phic, inclination]
)


In [ ]:
fs = 4096
N = strain.shape[0]
df = fs / N
freqs_all = jnp.fft.rfftfreq(N, d=1.0 / fs)
band = (freqs_all >= 20) & (freqs_all <= 2048)
f_ref = 20


In [ ]:
# Generate ripple template
from ripple.waveforms.IMRPhenomD import gen_IMRPhenomD_hphc

fplus = float(arrays[f"antenna/{ifo}/fplus"])
fcross = float(arrays[f"antenna/{ifo}/fcross"])
hp, hc = gen_IMRPhenomD_hphc(
    jnp.asarray(freqs_all[band], dtype=jnp.float64),
    theta_truth,
    float(f_ref),
)
hp = hp.astype(jnp.complex128)
hc = hc.astype(jnp.complex128)

h_band = fplus * hp + fcross * hc

# Embed the band-limited template back into the full rFFT grid (zeros outside)
h_full = jnp.zeros_like(freqs_all, dtype=jnp.complex128)
h_full = h_full.at[band].set(h_band)

# Physicist's convention: h̃(f) = Δt · FFT(h)
d_full = jnp.fft.rfft(strain) / fs

psd_safe = jnp.where(band, psd_arr, jnp.inf)


In [ ]:
inner_hh = 4.0 * df * jnp.real(jnp.sum(h_full * jnp.conj(h_full) / psd_safe))
optimal_snr = jnp.sqrt(inner_hh)
truth_snr = float(arrays[f"truth/snr_{ifo}"])
print(f"Optimal SNR (sqrt<h|h>): {float(optimal_snr):.3f}")
print(f"True SNR:                {truth_snr:.3f}")


In [13]:
from ml4gw.transforms import SpectralDensity
import inspect
print(inspect.getsource(SpectralDensity))

class SpectralDensity(torch.nn.Module):
    """
    Transform for computing either the power spectral density
    of a batch of multichannel timeseries, or the cross spectral
    density of two batches of multichannel timeseries.

    On ``SpectralDensity.forward`` call, if only one tensor is provided,
    this transform will compute its power spectral density. If a second
    tensor is provided, the cross spectral density between the two
    timeseries will be computed. For information about the allowed
    relationships between these two tensors, see the documentation to
    :meth:`~ml4gw.spectral.fast_spectral_density`.

    Note that the cross spectral density computation is currently
    only available for :meth:`~ml4gw.spectral.fast_spectral_density`. If
    ``fast=False`` and a second tensor is passed to ``SpectralDensity.forward``,  # noqa E501
    a ``NotImplementedError`` will be raised.

    Args:
        sample_rate:
            Rate at which tensors passed to ``forward`` w

In [ ]:
import jax
import jax.numpy as jnp
from jax import lax

# ---- parameters (match the SpectralDensity(...) call in injections.py) ----
sample_rate = 4096.0          # must match the data's sample rate
fftlength   = 2.0             # seconds
overlap     = None            # defaults to fftlength / 2
average     = "mean"          # "mean" or "median"

if overlap is None:
    overlap = fftlength / 2
elif overlap >= fftlength:
    raise ValueError("overlap must be < fftlength")

nperseg = int(fftlength * sample_rate)
nstride = nperseg - int(overlap * sample_rate)

# hanning window — torch.hann_window(N) is periodic of length N:
# w[n] = 0.5 - 0.5*cos(2*pi*n/N), n=0..N-1
n_idx = jnp.arange(nperseg, dtype=jnp.float64)
window = 0.5 - 0.5 * jnp.cos(2.0 * jnp.pi * n_idx / nperseg)

# "density" normalization, identical to SpectralDensity.__init__
scale = 1.0 / (sample_rate * jnp.sum(window ** 2))


# ---- scipy-style median with bias correction (matches ml4gw.spectral.median) ----
def _scipy_median(x, axis):
    n = x.shape[axis]
    ii_2 = 2.0 * jnp.arange(1.0, (n - 1) // 2 + 1)
    bias = 1.0 + jnp.sum(1.0 / (ii_2 + 1.0) - 1.0 / ii_2)
    return jnp.median(x, axis=axis) / bias


# ---- unfold the last axis into overlapping windows ----
def _unfold(x, nperseg, nstride):
    """x: (..., T) -> (..., num_segments, nperseg)"""
    T = x.shape[-1]
    num_segments = (T - nperseg) // nstride + 1
    starts = jnp.arange(num_segments) * nstride

    def take(s):
        return lax.dynamic_slice_in_dim(x, s, nperseg, axis=-1)

    framed = jax.vmap(take)(starts)            # (num_segments, ..., nperseg)
    return jnp.moveaxis(framed, 0, -2)         # (..., num_segments, nperseg)


# ---- slow/exact spectral_density, faithful to ml4gw ----
def spectral_density(x, *, nperseg, nstride, window, scale, average="mean"):
    x = x.astype(jnp.float64)

    segs = _unfold(x, nperseg, nstride)        # (..., num_segments, nperseg)
    segs = segs - segs.mean(axis=-1, keepdims=True)
    segs = segs * window

    fft = jnp.abs(jnp.fft.rfft(segs, axis=-1)) ** 2

    # one-sided correction: double interior bins.
    if nperseg % 2:
        fft = fft.at[..., 1:].multiply(2.0)
    else:
        fft = fft.at[..., 1:-1].multiply(2.0)

    fft = fft * scale

    if average == "mean":
        return fft.mean(axis=-2)
    elif average == "median":
        return _scipy_median(fft, axis=-2)
    else:
        raise ValueError(f'average must be "mean" or "median", got {average!r}')


In [ ]:
# ---- Matched filtering with the JAX-computed PSD ----------------------------
# Estimate the one-sided PSD straight from the loaded strain via the JAX
# spectral_density above, then run the proper matched filter:
#
#     z(t) = 4 ∫ conj(d̃) h̃ e^{2πift} / S_n(f) df
#     ρ(t) = |z(t)| / sqrt(<h|h>)
#
# d_full and h_full live on the full rFFT grid of the segment; the JAX PSD is
# estimated on its own (shorter) Welch grid and then interpolated onto that
# grid so the integrand is a single product over freqs_all.

psd_jax_short = spectral_density(
    strain,
    nperseg=nperseg,
    nstride=nstride,
    window=window,
    scale=scale,
    average=average,
)
freqs_short = jnp.fft.rfftfreq(nperseg, d=1.0 / sample_rate)

# Interpolate onto the segment's rFFT grid and mask out-of-band bins.
psd_jax = jnp.interp(freqs_all, freqs_short, psd_jax_short)
psd_jax_safe = jnp.where(band, psd_jax, jnp.inf)

# <h|h> with the JAX PSD (sanity check that it tracks the stored PSD)
hh_jax = 4.0 * df * jnp.real(jnp.sum(h_full * jnp.conj(h_full) / psd_jax_safe))

# Matched-filter time series. Zero-pad the one-sided integrand back to a full
# length-N complex spectrum (negative freqs = 0) before the inverse FFT, so the
# 4 · fs factor matches the continuous-time normalisation.
integrand = jnp.conj(d_full) * h_full / psd_jax_safe
full = jnp.zeros(N, dtype=jnp.complex128).at[: integrand.shape[0]].set(integrand)
z_t = 4.0 * fs * jnp.fft.ifft(full)

rho_t = jnp.abs(z_t) / jnp.sqrt(hh_jax)
peak_idx = int(jnp.argmax(rho_t))
peak_snr = float(rho_t[peak_idx])
peak_time = peak_idx / fs

print(f"JAX <h|h>           : {float(hh_jax):.3f}")
print(f"Stored-PSD <h|h>    : {float(inner_hh):.3f}")
print(f"Matched-filter peak : {peak_snr:.3f}  at t = {peak_time:.4f} s")
print(f"True {ifo} SNR         : {truth_snr:.3f}")
